In [1]:
import requests
import time
import csv
import json

HEADERS = {
    "User-Agent": "academic-research-script/1.0"
}

REQUEST_DELAY = 1.2
POSTS_PER_PERIOD = 80   # ~240 total

BOT_PHRASES = [
    "i am a bot",
    "performed automatically",
    "contact the moderators",
    "discord server",
    "this action was performed",
]

# ---------------------------
# TIME PERIODS (UTC)
# ---------------------------
PERIODS = {
    "during_covid": (1583020800, 1625097600),
    "post_covid":   (1625097600, 1672444800),
    "ai_era":       (1704067200, 1735689600),
}

# ---------------------------
# SUBREDDITS
# ---------------------------
SUBREDDITS = [
    "developersIndia",
    "cscareerquestions",
    "ExperiencedDevs",
    "ITCareerQuestions",
    "programming",
    "learnprogramming",
    "careerguidance",
    "jobs",
    "recruitinghell",
    "antiwork",
    "technology",
    "artificial",
    "MachineLearning",
    "datascience"
]

# ---------------------------
# HELPERS
# ---------------------------
def is_bot_comment(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in BOT_PHRASES)

def fetch_posts(subreddit, after, before, size=100):
    url = "https://api.pullpush.io/reddit/search/submission/"
    params = {
        "subreddit": subreddit,
        "after": after,
        "before": before,
        "size": size,
        "sort": "desc",
        "sort_type": "created_utc",
    }
    res = requests.get(url, params=params, headers=HEADERS, timeout=20)
    res.raise_for_status()
    return res.json().get("data", [])

def fetch_comments(post_id):
    url = "https://api.pullpush.io/reddit/comment/search/"
    params = {
        "link_id": post_id,
        "size": 10,
        "sort": "asc"
    }
    res = requests.get(url, params=params, headers=HEADERS, timeout=20)
    res.raise_for_status()

    comments = []
    for c in res.json().get("data", []):
        body = c.get("body", "").strip()
        if body and not is_bot_comment(body):
            comments.append(body)
        if len(comments) == 2:
            break

    return comments

# ---------------------------
# MAIN COLLECTION
# ---------------------------
rows = []
doc_id = 0

for period, (after, before) in PERIODS.items():
    collected = 0
    print(f"\nCollecting period: {period}")

    for subreddit in SUBREDDITS:
        if collected >= POSTS_PER_PERIOD:
            break

        print(f"  r/{subreddit}")
        posts = fetch_posts(subreddit, after, before)

        for post in posts:
            if collected >= POSTS_PER_PERIOD:
                break

            title = post.get("title", "")
            body = post.get("selftext", "")
            post_text = (title + " " + body).strip()

            if len(post_text) < 150:
                continue

            comments = fetch_comments(post["id"])
            if len(comments) < 1:
                continue

            record = {
                "doc_id": doc_id,
                "period": period,
                "subreddit": subreddit,
                "post_text": post_text,
                "comment_1": comments[0],
                "comment_2": comments[1] if len(comments) > 1 else ""
            }

            rows.append(record)
            doc_id += 1
            collected += 1
            time.sleep(REQUEST_DELAY)

# ---------------------------
# SAVE CSV
# ---------------------------
with open("reddit_temporal_raw.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "doc_id",
            "period",
            "subreddit",
            "post_text",
            "comment_1",
            "comment_2"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

# ---------------------------
# SAVE JSON
# ---------------------------
with open("reddit_temporal_raw2.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(rows)} documents")
print("Files created:")
print(" - reddit_temporal_raw.csv")
print(" - reddit_temporal_raw.json")



  r/developersIndia
  r/cscareerquestions

  r/developersIndia
  r/cscareerquestions
  r/ExperiencedDevs

  r/developersIndia
  r/cscareerquestions
  r/ExperiencedDevs

Saved 240 documents
Files created:
 - reddit_temporal_raw.csv
 - reddit_temporal_raw.json
